# 3D Delaunay clustering: `dtscan_3d`

3D generalization of DTSCAN:

> Kim, J., & Cho, J. (2019). Delaunay Triangulation-Based Spatial Clustering Technique for Enhanced Adjacent Boundary Detection and Segmentation of LiDAR 3D Point Clouds. *Sensors*, 19(18), 3926.

`dtscan.hpp`'s own `dtscan()` implements the paper exactly as published, over a 2D Delaunay triangulation: triangles are pruned by a z-score on their own area and on their longest edge's length, then DBSCAN's region-growing runs over the surviving triangulation-adjacency graph. `concavehull.three_d.dtscan_3d` is the direct 3D analogue: triangles become tetrahedra from a true 3D Delaunay triangulation, "area" generalizes to "volume", and edge-length pruning plus the DBSCAN stage are otherwise unchanged.

Since this project's own `delaunator` triangulator is 2D-only, `dtscan_3d` builds its Delaunay triangulation via [GUDHI](https://github.com/GUDHI/gudhi-devel)'s `AlphaComplex` (CGAL's Delaunay triangulation under the hood) with `max_alpha_square=inf`, which includes every simplex -- i.e. the full, unfiltered Delaunay triangulation, not an alpha-shape subset.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers the '3d' projection
from ipywidgets import interact, IntSlider, FloatSlider

from concavehull.three_d import dtscan_3d


def plot_labeled_points_3d(ax, points, labels, title=None):
    """Noise (label -1) in light gray; each cluster in its own tab10 color."""
    colors = plt.cm.tab10.colors
    noise = labels == -1
    ax.scatter(points[noise, 0], points[noise, 1], points[noise, 2], s=10, color="lightgray", alpha=0.4,
               label="noise")
    for cid in sorted(set(labels.tolist()) - {-1}):
        mask = labels == cid
        ax.scatter(points[mask, 0], points[mask, 1], points[mask, 2], s=14,
                   color=colors[cid % len(colors)], label=f"cluster {cid}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    if title:
        ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)

## Example 1: Separated blobs

Four Gaussian blobs placed around 3D space, each one forming its own clean cluster -- the direct 3D analogue of `dtscan.hpp`'s own `test_three_separated_blobs_form_three_clean_clusters`.

In [2]:
@interact(
    n_per_blob=IntSlider(value=60, min=20, max=200, step=10, description="n_per_blob", continuous_update=False),
    separation=FloatSlider(value=15.0, min=5.0, max=25.0, step=1.0, description="separation", continuous_update=False),
    volume_z_threshold=FloatSlider(value=1.0, min=-1.0, max=3.0, step=0.1, description="volume_z_thr",
                                    continuous_update=False),
    length_z_threshold=FloatSlider(value=1.0, min=-1.0, max=3.0, step=0.1, description="length_z_thr",
                                    continuous_update=False),
    min_pts=IntSlider(value=4, min=1, max=10, description="min_pts", continuous_update=False),
    seed=IntSlider(value=1, min=0, max=20, description="seed", continuous_update=False),
)
def show_separated_blobs(n_per_blob, separation, volume_z_threshold, length_z_threshold, min_pts, seed):
    rng = np.random.default_rng(seed)
    centers = [(0, 0, 0), (separation, 0, 0), (separation / 2, separation * 0.87, 0),
               (separation / 2, separation * 0.29, separation * 0.82)]
    pts = np.vstack([rng.normal(c, 0.6, size=(n_per_blob, 3)) for c in centers])

    r = dtscan_3d(pts, volume_z_threshold=volume_z_threshold, length_z_threshold=length_z_threshold,
                  min_pts=min_pts)

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection="3d")
    plot_labeled_points_3d(ax, pts, r.labels, title=f"dtscan_3d -> {r.n_clusters} cluster(s)")
    plt.show()

interactive(children=(IntSlider(value=60, continuous_update=False, description='n_per_blob', max=200, min=20, …

## Example 2: Resisting the chaining problem

Two dense blobs joined by a thin bridge of sparse points -- the exact "touching problem" (Figure 2 of the paper) DTSCAN's z-score pruning + triangulation-adjacency neighbor rule is designed to survive, unlike a naive single-linkage/distance-threshold clustering, which would merge the two blobs via the chain. Push `volume_z_thr`/`length_z_thr` up to see the bridge eventually get accepted and the two blobs merge into one cluster -- the same threshold-sensitivity `dtscan.hpp`'s own 2D version has.

In [3]:
@interact(
    n_per_blob=IntSlider(value=60, min=20, max=200, step=10, description="n_per_blob", continuous_update=False),
    blob_separation=FloatSlider(value=12.0, min=6.0, max=20.0, step=1.0, description="blob_sep",
                                 continuous_update=False),
    n_bridge=IntSlider(value=5, min=2, max=15, description="n_bridge", continuous_update=False),
    volume_z_threshold=FloatSlider(value=1.0, min=-1.0, max=3.0, step=0.1, description="volume_z_thr",
                                    continuous_update=False),
    length_z_threshold=FloatSlider(value=1.0, min=-1.0, max=3.0, step=0.1, description="length_z_thr",
                                    continuous_update=False),
    min_pts=IntSlider(value=4, min=1, max=10, description="min_pts", continuous_update=False),
)
def show_chaining_resistance(n_per_blob, blob_separation, n_bridge, volume_z_threshold, length_z_threshold, min_pts):
    rng = np.random.default_rng(2)
    blob_a = rng.normal([0, 0, 0], 0.4, size=(n_per_blob, 3))
    blob_b = rng.normal([blob_separation, 0, 0], 0.4, size=(n_per_blob, 3))
    bridge = np.array([[blob_separation * i / (n_bridge + 1), 0.0, 0.0] for i in range(1, n_bridge + 1)])
    pts = np.vstack([blob_a, blob_b, bridge])

    r = dtscan_3d(pts, volume_z_threshold=volume_z_threshold, length_z_threshold=length_z_threshold,
                  min_pts=min_pts)

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection="3d")
    label_a, label_b = r.labels[0], r.labels[n_per_blob]
    status = "merged into one cluster" if label_a == label_b and label_a != -1 else "kept as separate clusters"
    plot_labeled_points_3d(ax, pts, r.labels, title=f"dtscan_3d -> {r.n_clusters} cluster(s) ({status})")
    plt.show()

interactive(children=(IntSlider(value=60, continuous_update=False, description='n_per_blob', max=200, min=20, …

## Example 3: Noise rejection on a uniformly scattered cloud

A roughly homogeneous random point cloud has no real multi-density structure, so -- just like `dtscan.hpp`'s own `test_sparse_scattered_points_are_mostly_noise_at_an_aggressive_threshold` -- a meaningfully negative threshold (an aggressive, below-average cut) is needed to force most of it to noise; a loose/default threshold instead keeps most points in one big cluster, since a Delaunay triangulation gives every point many neighbors regardless of how "random" the layout is.

In [4]:
@interact(
    n_points=IntSlider(value=150, min=30, max=400, step=10, description="n_points", continuous_update=False),
    volume_z_threshold=FloatSlider(value=-0.3, min=-2.0, max=2.0, step=0.1, description="volume_z_thr",
                                    continuous_update=False),
    length_z_threshold=FloatSlider(value=-0.3, min=-2.0, max=2.0, step=0.1, description="length_z_thr",
                                    continuous_update=False),
    min_pts=IntSlider(value=4, min=1, max=10, description="min_pts", continuous_update=False),
    seed=IntSlider(value=3, min=0, max=20, description="seed", continuous_update=False),
)
def show_uniform_scatter(n_points, volume_z_threshold, length_z_threshold, min_pts, seed):
    rng = np.random.default_rng(seed)
    pts = rng.uniform(-25, 25, size=(n_points, 3))

    r = dtscan_3d(pts, volume_z_threshold=volume_z_threshold, length_z_threshold=length_z_threshold,
                  min_pts=min_pts)
    n_noise = int((r.labels == -1).sum())

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection="3d")
    plot_labeled_points_3d(ax, pts, r.labels,
                           title=f"dtscan_3d -> {r.n_clusters} cluster(s), {n_noise}/{n_points} noise")
    plt.show()

interactive(children=(IntSlider(value=150, continuous_update=False, description='n_points', max=400, min=30, s…

## Notes

- `dtscan_3d` needs at least 4 points (a single tetrahedron); fewer returns every point as noise (`n_clusters=0`) without calling into GUDHI.
- `volume_z_threshold`/`length_z_threshold` are z-scores computed over the *whole* triangulation's own tetrahedron-volume/edge-length distributions (matching the paper's own global, not per-region, computation) -- there's no single universal good default independent of the point cloud's own scale and density, same as `chi` for chi-outline.
- See `concavehull/three_d/dtscan_3d.py` for the implementation, and `tests/test_dtscan_3d.py` for the full correctness-property test suite this notebook's examples are drawn from.